# Hotel Booking Demand — Starter Notebook

119,390 real bookings for two hotels in Portugal (a resort hotel in the Algarve and a city hotel in Lisbon) with arrival dates from July 2015 to August 2017 (Antonio, de Almeida & Nunes, 2019). Each row is **one booking**: when it was made, who made it, what was requested, the price, and whether it was **canceled**.

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | The main outcome and predictor variables |

This notebook is for **getting to know the dataset**. Every cell is starter code (🧰) written by the instructor; run them in order and read the output. You do your homework analysis in the separate homework notebooks, not here.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/02_hotels/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/02_hotels/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/02_hotels/"
DATA_URL = BASE_URL + "hotels.csv"

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
df = pd.read_csv(DATA_URL)
print("Loaded", df.shape[0], "bookings")

## Preprocessing: dates

The arrival date is split across three columns. We combine them, then work backwards to the date the booking was made (arrival date minus `lead_time` days).

In [ ]:
month_number = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, "July": 7,
                "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}
df["arrival_date"] = pd.to_datetime(pd.DataFrame({"year": df["arrival_date_year"],
                                                  "month": df["arrival_date_month"].map(month_number),
                                                  "day": df["arrival_date_day_of_month"]}))
df["booking_date"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])

# Booking agent and company are ID numbers; missing means "none". We store them as text labels.
df["agent"] = df["agent"].fillna(0).astype(int).astype(str).replace("0", "none")
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df[["hotel", "booking_date", "arrival_date", "lead_time", "is_canceled", "deposit_type"]].head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Numeric summaries for the most useful columns
df[["is_canceled", "lead_time", "adr", "total_nights", "adults", "children",
    "total_of_special_requests", "booking_changes", "previous_cancellations"]].describe().round(2)

In [ ]:
# Categorical columns: how many bookings in each category
for col in ["hotel", "deposit_type", "customer_type", "market_segment", "meal"]:
    print(df[col].value_counts(), "\n")

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** `company` is missing for most bookings (most guests aren't booking through a company), and `agent` was missing for about 14% (now labeled `none`). `country` is missing for a few hundred bookings. `adr` (average daily rate, in euros) has one negative value and one value of 5,400, which are almost certainly errors.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# Cancellation rate by hotel
df.groupby("hotel")["is_canceled"].mean().plot(kind="bar")
plt.ylabel("Proportion canceled")
plt.title("Cancellations by hotel")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Lead time (days between booking and arrival) for canceled vs. kept bookings
plt.hist(df.loc[df["is_canceled"] == 0, "lead_time"], bins=60, alpha=0.6, label="kept", density=True)
plt.hist(df.loc[df["is_canceled"] == 1, "lead_time"], bins=60, alpha=0.6, label="canceled", density=True)
plt.xlabel("Lead time (days)")
plt.ylabel("Density")
plt.legend()
plt.title("How far ahead people booked")
plt.show()

In [ ]:
# Cancellation rate by deposit type (look closely at "Non Refund": is it what you expected?)
print(df.groupby("deposit_type")["is_canceled"].agg(["mean", "count"]).round(3))

In [ ]:
# Bookings by arrival month
month_order = list(month_number.keys())
df["arrival_date_month"].value_counts().reindex(month_order).plot(kind="bar")
plt.ylabel("Number of bookings")
plt.title("Bookings by arrival month")
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `is_canceled` (yes/no), `adr` (price per night), `total_of_special_requests` (count), `booking_changes` (count).

**Predictors:** `lead_time`, `hotel`, `deposit_type`, `market_segment`, `customer_type`, `is_repeated_guest`, `previous_cancellations`, `adults`, `children`, arrival month.

In [ ]:
outcome_vars = ["is_canceled", "adr", "total_of_special_requests", "booking_changes"]
predictor_vars = ["lead_time", "is_repeated_guest", "previous_cancellations", "adults", "total_nights"]
df[outcome_vars + predictor_vars].describe().round(2)